---
title: Sampler inputs and outputs
description: Understand the input and output format of Sampler primitives
---


# Sampler inputs and outputs

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.5.2
qiskit-ibm-runtime~=0.47.0
```
</AccordionItem>
</Accordion>

This page gives an overview of the inputs and outputs of the `qiskit-ibm-runtime` Sampler primitive, which executes workloads on the IBM Quantum&reg; Compute Service. Sampler lets you efficiently define vectorized workloads by using a data structure known as a [**Primitive Unified Bloc (PUB)**](/docs/qiskit/primitive-input-output#pubs). They are used as inputs to the [`run()`](/docs/api/qiskit-ibm-runtime/executor-sampler-sampler#run) method for the Sampler primitive, which executes the defined workload as a job. Then, after the job has completed, the results are returned in a format that is dependent on both the PUBs used as well as the runtime options specified from the primitive.

## Inputs

Each PUB is in the format:

(`<single circuit>`, `<one or more optional parameter value>`, `<optional shots>`),

There can be multiple `parameter values` items, and each item can be either an array or a single parameter, depending on the chosen circuit. Additionally, the input must contain measurements.

For the Sampler primitive, a PUB can contain at most three values:
- A single `QuantumCircuit`, which may contain one or more [`Parameter`](/docs/api/qiskit/qiskit.circuit.Parameter) objects
   *Note: These circuits should also include measurement instructions for each of the qubits to be sampled.*
- A collection of parameter values to bind the circuit against $\theta_k$ (only needed if any `Parameter` objects are used that must be bound at runtime)
- (Optionally) a number of shots to measure the circuit with

---

The following code demonstrates an example set of vectorized inputs to the `Sampler` primitive and executes them on an IBM&reg; backend as a single `RuntimeJobV2 ` object.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray

from qiskit_ibm_runtime import QiskitRuntimeService

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Sampler

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_sampler import Sampler

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Sampler object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = Sampler(mode=backend)
job = sampler.run([sampler_pub])
result = job.result()

## Outputs

After one or more PUBs are sent to a QPU for execution and a job successfully completes, the data is returned as a [`PrimitiveResult`](/docs/api/qiskit/qiskit.primitives.PrimitiveResult) container object accessed by calling the `RuntimeJobV2.result()` method. The `PrimitiveResult` contains an iterable list of [`SamplerPubResult`](/docs/api/qiskit/qiskit.primitives.SamplerPubResult) objects that contain the execution results for each PUB. These data are samples of the circuit output.

Each element of this list corresponds to a PUB submitted to the primitive's `run()` method (for example, a job submitted with 20 PUBs will return a `PrimitiveResult` object that contains a list of 20 `SamplerPubResult` objects, one corresponding to each PUB).

Each `SamplerPubResult` object possesses both a `data` and a `metadata` attribute.
    - The `data` attribute is a customized [`DataBin`](/docs/api/qiskit/qiskit.primitives.DataBin) that contains the actual measurement values, standard deviations, and so forth. The data bins are dict-like objects that contain one `BitArray` per `ClassicalRegister` in the circuit.
    - The `BitArray` class is a container for ordered shot data. It stores the sampled bitstrings as bytes inside a two-dimensional array. The left-most axis of this array runs over ordered shots, while the right-most axis runs over bytes.
    - The `metadata` attribute contains information about the runtime options used (explained later in the [Result metadata](#result-metadata) section of this page).


The following is a visual outline of the `PrimitiveResult` data structure:

```
    └── PrimitiveResult
        ├── SamplerPubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── SamplerPubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second pub
        ├── ...
        ├── ...
        └── ...
```

Put simply, a single job returns a [`PrimitiveResult`](/docs/api/qiskit/qiskit.primitives.PrimitiveResult) object and contains a list of one or more [`SamplerPubResult`](/docs/api/qiskit/qiskit.primitives.SamplerPubResult) objects. These `SamplerPubResult` objects then store the measurement data for each PUB that was submitted to the job.

As a first example, let us look at the following ten-qubit circuit:

In [2]:
# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=4096, num_bits=10>)

The shape of register `meas` is (4096, 2).

The bytes in register `alpha`, shot by shot:
[[  3 255]
 [  0 107]
 [  2 172]
 ...
 [  1 163]
 [  0 236]
 [  3 255]]



It can sometimes be convenient to convert away from the bytes format in the `BitArray` to bitstrings. The `get_count` method returns a dictionary mapping bitstrings to the number of times that they occurred.

In [3]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'1111111111': 31, '0001101011': 13, '1010101100': 7, '1100011011': 9, '1011010101': 24, '0010110111': 7, '0011101100': 6, '1010110001': 10, '0000000000': 25, '1100011001': 7, '0110000010': 8, '1110011011': 14, '0011010100': 27, '0101000111': 9, '0101011100': 9, '0000111111': 11, '1111110001': 24, '1011100011': 5, '0101010001': 1, '0100101011': 22, '0011101011': 6, '0101000000': 9, '1110011001': 6, '0001001110': 10, '0111001111': 8, '0100010100': 7, '0001110010': 11, '1100110101': 10, '0100101010': 30, '1101001000': 9, '1101110010': 6, '1000001110': 20, '0001110110': 8, '0100111011': 8, '0100011100': 8, '0001011000': 6, '1101010111': 11, '0000111110': 9, '0001000000': 8, '0011100010': 4, '0101100111': 4, '1001011001': 5, '1001111000': 14, '1000011111': 4, '0000001111': 33, '0110011011': 11, '1001110011': 7, '1100101010': 29, '0011011011': 20, '0111001100': 5, '1001110111': 11, '0111111001': 2, '0001000001': 1, '1001111101': 11, '1100111010': 4, '0100100101': 22, '1111000000': 1

When a circuit contains more than one classical register, the results are stored in different `BitArray` objects. The following example modifies the previous snippet by splitting the classical register into two distinct registers:

In [4]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=4096, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=4096, num_bits=9>)


### Use `BitArray` objects for performant post-processing

Since arrays generally offer better performance compared to dictionaries, it is advisable to perform any post-processing directly on the `BitArray` objects rather than on dictionaries of counts. The `BitArray` class offers a range of methods to perform some common post-processing operations:

In [5]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == 0b1
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (4096, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [0]
 [1]
 ...
 [1]
 [1]
 [1]]

The shape of register `beta` is (4096, 2).
The bytes in register `beta`, shot by shot:
[[  1 255]
 [  1 142]
 [  1  59]
 ...
 [  0  88]
 [  1 196]
 [  1 255]]

The shape of `beta` after post-selection is (2034, 2).
The bytes in `beta` after post-selection:
[[  1 255]
 [  1  59]
 [  1 106]
 ...
 [  0  88]
 [  1 196]
 [  1 255]]
The shape of `beta` after bit-wise slicing is (4096, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [6]
 [3]
 ...
 [0]
 [4]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  1 255]
 [  1 142]
 [  1  59]
 [  1 106]
 [  0  50]]

Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: 0.00146484375
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: 0.01123046875

The shape of the merged results is (409

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `SamplerPubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `SamplerPubResult.metadata`.

The client-side Sampler submits its workload as an [Executor](/docs/ibm-quantum-compute/executor-input-output) job. The metadata of the underlying Executor result, which includes execution timing information called [_execution spans_](#execution-spans), is available under the `"executor"` key of `PrimitiveResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [6]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'executor' : Metadata(chunk_timing=[ChunkSpan(start=datetime.datetime(2026, 11, 18, 21, 46, 24, 896045, tzinfo=datetime.timezone.utc), stop=datetime.datetime(2026, 11, 18, 21, 46, 24, 965016, tzinfo=datetime.timezone.utc), parts=[ChunkPart(idx_item=0, size=1)])]),
'options' : {'default_shots': 4096, 'dynamical_decoupling': {'enable': False, 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap', 'sequence_type': 'XX', 'skip_reset_qubits': False}, 'environment': {'image': None, 'job_tags': [], 'log_level': 'WARNING', 'max_execution_time': None, 'private': False}, 'execution': {'init_qubits': True, 'meas_type': 'classified', 'rep_delay': None, 'scheduler_timing': False, 'stretch_values': False}, 'experimental': {}, 'max_execution_time': None, 'simulator': {'angle_decimals': 5, 'layer_noise_model': None, 'seed_simulator': None, 'warn_absent': True}, 'twirling': {'enable_gates': False, 'enable_measure': False, 'group': 'pauli', 'num_random

<span id="execution-spans"></span>
### View execution spans

The client-side Sampler uses Executor to run jobs, which executes its work on the QPU in _chunks_. The timing of each chunk is in `result.metadata["executor"].chunk_timing`, a list of [`ChunkSpan`](/docs/api/qiskit-ibm-runtime/results-chunk-span) objects. Each `ChunkSpan` has `start` and `stop` timestamps and a list of `parts`, where each part gives an item index (`idx_item`) and the number of configurations from that item (`size`) that ran in the chunk.

Sampler creates one Executor program item per PUB, so `idx_item` is the index of the PUB, in the order the PUBs were given to `run()`. If twirling is enabled, each randomization of each parameter value set counts as a separate configuration.

To visualize the timings, pass the list to [`draw_chunk_timings`](/docs/api/qiskit-ibm-runtime/visualization-draw-chunk-timings), as shown in the following example.

To learn what the chunk timestamps mean, how to read the plot, and why gaps can appear between chunks, see [View execution spans](/docs/ibm-quantum-compute/executor-input-output#execution-spans) in the Executor inputs and outputs guide.

In [7]:
# Define a circuit with one parameter.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

# Define two PUBs, each with a different set of parameter values
sampler_pubs = [
    (transpiled_circuit, np.random.uniform(size=(10, 1))),
    (transpiled_circuit, np.random.uniform(size=(20, 1))),
]

job = sampler.run(sampler_pubs, shots=1024)
result = job.result()

In [8]:
from qiskit_ibm_runtime.visualization import draw_chunk_timings

# Note the "executor" key: the timing comes from the underlying Executor job
chunk_timing = result.metadata["executor"].chunk_timing

for idx, span in enumerate(chunk_timing):
    duration = (span.stop - span.start).total_seconds()
    print(f"Chunk {idx}: {span.start} to {span.stop} ({duration:.3f} s)")
    for part in span.parts:
        print(f"  PUB {part.idx_item}: {part.size} configurations")

fig = draw_chunk_timings(chunk_timing, names="Sampler job")

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("chunk_timings.html")

Chunk 0: 2026-11-18 21:46:32.449204+00:00 to 2026-11-18 21:46:32.493264+00:00 (0.044 s)
  PUB 0: 10 configurations
  PUB 1: 20 configurations


<span id="legacy-execution-spans"></span>
### Execution spans in the legacy Sampler

As of `qiskit-ibm-runtime` v0.50.0, the legacy server-side Sampler is deprecated. The client-side Sampler does not return `ExecutionSpans` objects; use the [chunk timing](#execution-spans) described previously instead. See [Migrate to client-side primitives](/docs/guides/migrate-to-client-side-primitives) for instructions to migrate.

    <Accordion>
    <AccordionItem title="**Click to learn more.**">
    The results of jobs from the legacy server-side [`Sampler`](/docs/api/qiskit-ibm-runtime/server-side-sampler-sampler-v2) contain execution timing information in `result.metadata["execution"]["execution_spans"]`, in the form of an [`ExecutionSpans`](/docs/api/qiskit-ibm-runtime/execution-span-execution-spans) object. This object is a list-like container of [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

    An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any PUB index, returns a boolean mask that is `True` for all shots executed during its window. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

    ```python
    from qiskit.primitives import BitArray

    spans = result.metadata["execution"]["execution_spans"]

    # Get the mask of the 1st PUB for the 0th span.
    mask = spans[0].mask(0)

    # Create a new bit array containing only the PUB-0 data collected during this span.
    bits = result[0].data.meas
    filtered_data = BitArray(bits.array[mask], bits.num_bits)

    # Take the subset of spans that reference data in PUBs 0 or 2
    spans.filter_by_pub([0, 2])

    # View global information about the collection of execution spans
    print("Number of execution spans:", len(spans))
    print("  Start of the first span:", spans.start)
    print("     End of the last span:", spans.stop)
    print("       Total duration (s):", spans.duration)
    ```

    To visualize legacy execution spans, use the [`draw_execution_spans`](/docs/api/qiskit-ibm-runtime/visualization-draw-execution-spans) function.

    It is possible for time windows specified by distinct execution spans to overlap. This is not because a QPU was performing multiple executions simultaneously, but is instead an artifact of certain classical processing that might happen concurrently with quantum execution. The guarantee is that the referenced data definitely occurred in the reported execution span, but not necessarily that the limits of the time window are as tight as possible.
  </AccordionItem>
</Accordion>